# Notebook 5: Knowledge base and retrieval for code embeddings (LLM + RAG)

This notebook builds, for every input and label code, the text that a frozen biomedical text encoder turns
into an embedding. Two sources are compared in Notebook 6:

* **desc**: the description of the code in its own code system (CCS path for diagnoses and procedures, ATC
  name for medications). This is the "LLM embedding" condition: the text encoder is the only source of
  outside knowledge.
* **rag**: for diagnosis categories, the description is augmented with knowledge *retrieved* from external
  resources: the Mondo diseases that fall into the category (rare diseases first), their most frequent HPO
  phenotypes and their genes. Retrieval is structured (through ICD-10-CM cross-references and the Notebook 1
  mapping), not similarity search, so that every retrieved item is attached to the right label.

Only public resources and code descriptions are used. No MIMIC-IV record leaves the cluster, and nothing
derived from MIMIC-IV frequencies enters the documents (no test-set leakage). The encoder defaults to
BioLORD-2023, a sentence encoder trained to align biomedical concept names with their definitions; set
`RAREPRED_ENCODER` to compare others, or `none` to only build the documents.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, collections, pickle
from rarepred.resources import (load_multilevel_ccs_from_rda, load_multilevel_ccs_from_hcup_csv, load_gem_first,
                                parse_mondo, load_hpo_annotations)
from rarepred.knowledge import CCSKnowledgeIndex, describe_ccs, pyhealth_names, encode_texts
ENCODER = os.environ.get('RAREPRED_ENCODER', 'FremyCompany/BioLORD-2023')
print('encoder:', ENCODER)

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

encoder: none


## 1. Resources

In [2]:
HCUP_CSV = ROOT / 'data' / 'mappings' / 'Multi_Level_CCS_2015' / 'ccs_multi_dx_tool_2015.csv'
ccs, ccs_labels = (load_multilevel_ccs_from_hcup_csv(HCUP_CSV) if HCUP_CSV.exists()
                   else load_multilevel_ccs_from_rda(EXTERNAL))
icd9_to_ccs = dict(zip(ccs.icd9, ccs.ccs_ext))
gem = load_gem_first(EXTERNAL / '2018_I10gem.txt')
mondo = parse_mondo(EXTERNAL / 'mondo' / 'src' / 'ontology' / 'mondo-edit.obo')
phen, genes, hpo_names = load_hpo_annotations(EXTERNAL / 'hpo' / 'phenotype.hpoa',
                                              EXTERNAL / 'hpo' / 'genes_to_phenotype.txt', EXTERNAL / 'hpo' / 'hp.obo')
index = CCSKnowledgeIndex(mondo, gem, icd9_to_ccs, phen, genes, hpo_names)
print(f"CCS categories with >= 1 retrieved Mondo disease: {len(index.by_ccs)} of {ccs.ccs_ext.nunique()}")

CCS categories with >= 1 retrieved Mondo disease: 423 of 589


## 2. Documents for the diagnosis categories

In [3]:
docs = []
for c in sorted(ccs.ccs_ext.unique()):
    desc = describe_ccs(c, ccs_labels)
    rag, r = index.document(c, desc)
    docs.append({'code': c, 'desc': desc, 'rag': rag, 'n_diseases': r['n_diseases'], 'n_rare': r['n_rare'],
                 'n_phenotypes': len(r['phenotypes']), 'n_genes': len(r['genes'])})
docs = pd.DataFrame(docs)
aug = docs.rag.str.len() > docs.desc.str.len() + len('Diagnosis category: .')
print(f"categories augmented by retrieval: {aug.sum()} of {len(docs)}")
print(f"document length (characters): desc median {docs.desc.str.len().median():.0f}, "
      f"rag median {docs.rag.str.len().median():.0f}, rag max {docs.rag.str.len().max()}")
with open(RESULTS / 'knowledge_docs_ccs.jsonl', 'w') as f:
    for rec in docs.to_dict(orient='records'):
        f.write(json.dumps(rec) + '\n')

categories augmented by retrieval: 423 of 589
document length (characters): desc median 108, rag median 643, rag max 1500


In [4]:
for c in ['7.1.1', '3.11.3', '6.4.1', '14.5.26']:
    row = docs.set_index('code').loc[c]
    print(f"--- {c} | {row.n_diseases} Mondo diseases ({row.n_rare} rare)")
    print('desc:', row.desc)
    print('rag :', row.rag[:700], '...' if len(row.rag) > 700 else '')
    print()

--- 7.1.1 | 0 Mondo diseases (0 rare)
desc: Diseases of the circulatory system > Hypertension > Essential hypertension
rag : Diagnosis category: Diseases of the circulatory system > Hypertension > Essential hypertension. 

--- 3.11.3 | 681 Mondo diseases (670 rare)
desc: Endocrine; nutritional; and metabolic diseases and immunity disorders > Other nutritional; endocrine; and metabolic disorders > Other and unspecified metabolic; nutritional; and endocrine disorders
rag : Diagnosis category: Endocrine; nutritional; and metabolic diseases and immunity disorders > Other nutritional; endocrine; and metabolic disorders > Other and unspecified metabolic; nutritional; and endocrine disorders. Includes conditions such as oculocerebrorenal syndrome; congenital disorder of deglycosylation 1; mucolipidosis type II; PMM2-congenital disorder of glycosylation; ALG12-congenital disorder of glycosylation; cerebrotendinous xanthomatosis; mitochondrial DNA depletion syndrome 13; mucopolysaccharidosis ty

The retrieved text is dominated by rare diseases for the congenital and metabolic categories, which is the
intended effect; for a category such as essential hypertension retrieval adds little, and its embedding stays
close to the description-only one. A limit of structured retrieval is visible in the long tail: a category
that hosts hundreds of rare diseases is summarised by eight of them, so the document describes the category
as a mixture rather than any single disease.

## 3. Medication and procedure descriptions (needs the processed MIMIC-IV vocabularies)

In [5]:
rx_codes, px_codes, dx_input_codes = [], [], []
seq_path = PROCESSED / 'patient_sequences.pkl'
if seq_path.exists():
    seqs = pickle.load(open(seq_path, 'rb'))
    rx, px, dx = set(), set(), set()
    for visits in seqs.values():
        for _, d, p, r in visits:
            dx.update(d); px.update(p); rx.update(r)
    rx_codes, px_codes, dx_input_codes = sorted(rx), sorted(px), sorted(dx)
    rx_names = pyhealth_names('ATC', rx_codes)
    px_names = pyhealth_names('CCSPROC', px_codes)
    print(f"Rx codes {len(rx_codes)} ({sum(rx_names[c] != c for c in rx_codes)} named) | "
          f"Px codes {len(px_codes)} ({sum(px_names[c] != c for c in px_codes)} named) | "
          f"Dx input codes {len(dx_input_codes)} ({len(set(dx_input_codes) - set(docs.code))} without a CCS entry)")
else:
    print('processed/patient_sequences.pkl not found: only diagnosis documents are built.')

processed/patient_sequences.pkl not found: only diagnosis documents are built.


## 4. Encode

In [6]:
if ENCODER == 'none':
    print('RAREPRED_ENCODER=none: encoding skipped (run on the cluster).')
else:
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    out = {'encoder': ENCODER, 'dx_codes': np.array(docs.code),
           'dx_desc': encode_texts(list(docs.desc), ENCODER, dev),
           'dx_rag': encode_texts(list(docs.rag), ENCODER, dev)}
    if rx_codes:
        out.update(rx_codes=np.array(rx_codes), rx_desc=encode_texts([f"Medication class: {rx_names[c]}" for c in rx_codes], ENCODER, dev),
                   px_codes=np.array(px_codes), px_desc=encode_texts([f"Procedure: {px_names[c]}" for c in px_codes], ENCODER, dev))
    tag = ENCODER.split('/')[-1]
    np.savez_compressed(PROCESSED / f'knowledge_embeddings_{tag}.npz', **out)
    print('saved', PROCESSED / f'knowledge_embeddings_{tag}.npz', {k: v.shape for k, v in out.items() if hasattr(v, 'shape')})

RAREPRED_ENCODER=none: encoding skipped (run on the cluster).


## 5. Intrinsic check: do the embeddings respect the CCS hierarchy?

For each diagnosis category we look at its nearest neighbour (cosine) and ask whether it lies in the same CCS
chapter. This is a sanity check of the encoder, not a result about prediction: an encoder that fails it would
inject noise rather than knowledge.

In [7]:
if ENCODER != 'none':
    chap = np.array([c.split('.')[0] for c in docs.code])
    hosts = docs.n_rare.values > 0
    rows = []
    for src in ['dx_desc', 'dx_rag']:
        Z = out[src]; S = Z @ Z.T; np.fill_diagonal(S, -np.inf)
        same = chap[S.argmax(1)] == chap
        rows.append({'source': src, 'nn same chapter (all)': same.mean(),
                     'hosting rare diseases': same[hosts].mean(), 'not hosting': same[~hosts].mean()})
    display(pd.DataFrame(rows).round(3))